<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/07-clasificacion-creditos/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 1.7 · Clasificación: ¿qué crédito revisamos primero?

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas cuatro celdas. Dejan listas `creditos`, `entreno`, `prueba`, `numeros` y `textos`. Deben quedar `5408` y `1531` filas.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [3]:
def preparar_creditos(creditos):
    """Agrega las variables del modelo y separa entrenamiento y prueba al 1 de julio de 2025."""
    tabla = creditos.assign(
        log_monto=np.log(creditos["monto_sobre_limite"]),
        tasa_previa=(creditos["morosos_previos"] / creditos["casos_previos"]).fillna(0),
        sin_historial=(creditos["casos_previos"] == 0).astype(int),
    )
    fecha = pd.to_datetime(tabla["fecha"])
    se_sabe = fecha + pd.to_timedelta(tabla["plazo_dias"] + 31, unit="D")
    con_resultado = tabla["moroso"].notna()
    corte = pd.Timestamp("2025-07-01")
    return tabla[(se_sabe <= corte) & con_resultado], tabla[(fecha >= corte) & con_resultado]

In [4]:
creditos = pd.read_csv(DATOS + "creditos.csv")

entreno, prueba = preparar_creditos(creditos)

numeros = ["log_monto", "antiguedad_anios", "tasa_previa", "sin_historial"]
textos = ["tipo"]

print(len(entreno), len(prueba))

5408 1531


### 1. ¿Fuga o no?

Responde en un comentario si cada columna se puede usar para predecir `moroso`, y por qué:

- a) `monto_sobre_limite`
- b) `dias_atraso`
- c) `morosos_previos`
- d) `tipo`

Y una más: hoy es 1 de julio. ¿Puedes entrenar con una compra del 20 de junio? ¿Por qué?

In [5]:
# a) Sí: el monto y el límite se conocen al vender.
# b) No: es fuga. El atraso se conoce después de la venta.
# c) Sí: solo cuenta compras cuyo resultado ya se conocía ese día.
# d) Sí: el tipo de cliente se conoce desde antes.

# No: su plazo no ha vencido, así que aún no se sabe si caerá en mora.

### 2. La tasa por tipo

Calcula la tasa de morosidad (`moroso` promedio) por `tipo` en `entreno`. ¿Qué tipo de cliente tiene la mayor?

In [6]:
entreno.groupby("tipo")["moroso"].mean().round(3)

tipo
Bodega         0.200
Minimarket     0.141
Restaurante    0.189
Name: moroso, dtype: float64

### 3. Tu modelo

Entrena la regresión logística, como en la clase, y guarda en `probabilidad` las de `prueba`. La más alta, redondeada a 3 decimales, debe dar `0.661`.

In [7]:
x_entreno = preparar_x(entreno, numeros, textos)
x_prueba = preparar_x(prueba, numeros, textos, x_entreno.columns)

modelo = make_pipeline(StandardScaler(), LogisticRegression())
modelo.fit(x_entreno, entreno["moroso"])

probabilidad = modelo.predict_proba(x_prueba)[:, 1]

print(round(probabilidad.max(), 3))

0.661


### 4. La lista de revisión

Crea una columna `probabilidad` en una copia de `prueba`, con `prueba.assign(probabilidad=probabilidad)`, y guárdala en `revision`. Ordénala de mayor a menor y muestra las 10 primeras, con `id_credito`, `id_cliente`, `monto` y `probabilidad`.

In [8]:
revision = prueba.assign(probabilidad=probabilidad)

revision.sort_values("probabilidad", ascending=False)[
    ["id_credito", "id_cliente", "monto", "probabilidad"]
].head(10)

,id_credito,id_cliente,monto,probabilidad
7093,K07094,C0183,928.0,0.660955
6041,K06042,C0238,310.0,0.627011
6634,K06635,C0210,92.0,0.558417
7351,K07352,C0058,672.0,0.516461
6833,K06834,C0250,580.0,0.510842
6816,K06817,C0058,584.0,0.502784
6304,K06305,C0238,92.0,0.495054
6842,K06843,C0058,504.0,0.486448
7311,K07312,C0058,420.0,0.468734
6188,K06189,C0107,816.0,0.468672


### 5. Otro umbral

¿Cuántas compras marca el modelo con un umbral de `0.3`? Deben ser `124`. ¿Y de esas, cuántas resultaron morosas? Deben ser `54`.

> 💡 Filtra `revision` con `revision["probabilidad"] >= 0.3` y suma `moroso`.

In [9]:
marcadas = revision[revision["probabilidad"] >= 0.3]

print(len(marcadas))
print(marcadas["moroso"].sum())

124
54.0


### 6. ¿Lo usarías así?

Responde en un comentario. La gerencia propone rechazar automáticamente a todo cliente con probabilidad mayor a 0.3. ¿Qué le dirías?

In [10]:
# No. Con ese umbral, más de la mitad de las marcadas (70 de 124) se habría pagado sin mora,
# y se rechazaría a buenos clientes. Además, el modelo no vio a clientes rechazados,
# y son datos sintéticos. Sirve para ordenar qué revisar: una persona decide.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 1.8: Cuando el 85 % de aciertos no sirve**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0